# Web Scraping de Financial Times

Este notebook contiene una versión **limpia, organizada y comentada** del código original.

El flujo se divide en dos fases:
1. Buscar artículos de una empresa en Financial Times y guardar sus títulos y URLs.
2. Visitar esas URLs, extraer información de cada artículo y generar un JSON por artículo.

Se han eliminado imports y funciones duplicadas, se han agrupado las configuraciones y se han añadido comentarios para explicar las partes importantes del código.

## 1. Imports y configuración general


In [ ]:
# Librerías estándar de Python
import datetime
import json
import os
import random
import time

# Selenium: permite controlar Chrome y localizar elementos de una página web.
from selenium import webdriver
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# Instala/gestiona automáticamente la versión de ChromeDriver necesaria.
from webdriver_manager.chrome import ChromeDriverManager

## 2. Datos auxiliares

Estos datos se utilizan para construir el JSON final de los artículos. Los autores, categorías, secciones y algunos valores son datos simulados.

In [ ]:
# Autores simulados que se asignarán aleatoriamente a los artículos.
AUTHORS = [
    {"id": "u1", "name": "Ana Pérez"},
    {"id": "u2", "name": "Luis Gómez"},
    {"id": "u3", "name": "Marta Sánchez"},
    {"id": "u4", "name": "Carlos Ruiz"},
    {"id": "u5", "name": "Laura Torres"},
]

CATEGORIES = [
    "Finanzas", "Tecnología", "Empresas", "Startups",
    "Economía", "Industria", "Innovación", "Marketing",
]

SECTIONS = [
    "Internacional", "Negocios", "Opinión", "Deportes",
    "Política", "Cultura", "Tecnología", "Economía",
]

# Palabras utilizadas como descripciones de las imágenes.
IMAGE_DESCRIPTIONS = [
    "vista", "ciudad", "personas", "noche", "noticia",
    "evento", "mundo", "historia", "negocios", "coche",
]

# Ruta de la extensión utilizada por el scraper.
# Modifica esta ruta si la extensión se encuentra en otra ubicación.
EXTENSION_PATH = r"C:\repositorio\WebScrapping\bypass-paywalls-chrome-clean-master"

## 3. Generación de User-Agents

Se genera una colección de User-Agents con diferentes plataformas y navegadores. Posteriormente se selecciona uno aleatoriamente al iniciar cada navegador.

In [ ]:
def generate_user_agents(count=200):
    """Genera una lista de User-Agents simulados."""

    platforms = [
        ("Windows NT 10.0; Win64; x64", "Chrome"),
        ("Macintosh; Intel Mac OS X 13_3_1", "Chrome"),
        ("Macintosh; Intel Mac OS X 12_6_5", "Safari"),
        ("X11; Linux x86_64", "Chrome"),
        ("X11; Linux x86_64", "Firefox"),
        ("iPhone; CPU iPhone OS 16_3 like Mac OS X", "Safari"),
        ("iPad; CPU OS 15_6 like Mac OS X", "Safari"),
        ("Linux; Android 13; SM-S918B", "Chrome Mobile"),
        ("Linux; Android 12; Pixel 6", "Chrome Mobile"),
    ]

    user_agents = []

    for _ in range(count):
        platform, browser = random.choice(platforms)

        if browser == "Chrome":
            version = (
                f"{random.randint(100, 123)}.0."
                f"{random.randint(1000, 5999)}.{random.randint(0, 150)}"
            )
            user_agent = (
                f"Mozilla/5.0 ({platform}) AppleWebKit/537.36 "
                f"(KHTML, like Gecko) Chrome/{version} Safari/537.36"
            )

        elif browser == "Firefox":
            version = f"{random.randint(80, 124)}.0"
            user_agent = (
                f"Mozilla/5.0 ({platform}; rv:{version}) "
                f"Gecko/20100101 Firefox/{version}"
            )

        elif browser == "Safari":
            major = random.randint(14, 17)
            minor = random.randint(0, 5)
            user_agent = (
                f"Mozilla/5.0 ({platform}) AppleWebKit/605.1.15 "
                f"(KHTML, like Gecko) Version/{major}.{minor} "
                "Mobile/15E148 Safari/604.1"
            )

        else:  # Chrome Mobile
            version = (
                f"{random.randint(100, 123)}.0."
                f"{random.randint(1000, 5999)}.{random.randint(0, 150)}"
            )
            user_agent = (
                f"Mozilla/5.0 ({platform}) AppleWebKit/537.36 "
                f"(KHTML, like Gecko) Chrome/{version} Mobile Safari/537.36"
            )

        user_agents.append(user_agent)

    return user_agents


# Se genera una sola vez y se reutiliza durante la ejecución del notebook.
USER_AGENTS = generate_user_agents()


def get_random_user_agent():
    """Devuelve un User-Agent seleccionado aleatoriamente."""
    return random.choice(USER_AGENTS)

## 4. Creación del navegador

Esta función configura Chrome, carga la extensión indicada y establece un User-Agent aleatorio. Los parámetros adicionales mantienen la configuración utilizada en el código original.

In [ ]:
def create_driver():
    """Crea y devuelve una instancia configurada de Chrome."""

    chrome_options = Options()

    # Carga la extensión de Chrome desde la ruta configurada.
    chrome_options.add_argument(f"--load-extension={EXTENSION_PATH}")
    chrome_options.add_argument(
        f"--disable-extensions-except={EXTENSION_PATH}"
    )
    chrome_options.add_argument("--allow-legacy-extension-manifests")

    # Opciones de configuración utilizadas por el scraper original.
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option(
        "excludeSwitches", ["enable-automation"]
    )
    chrome_options.add_experimental_option(
        "useAutomationExtension", False
    )
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # ChromeDriverManager descarga/selecciona el controlador necesario.
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    # Se establece el User-Agent mediante Chrome DevTools Protocol.
    user_agent = get_random_user_agent()
    driver.execute_cdp_cmd(
        "Network.setUserAgentOverride",
        {"userAgent": user_agent},
    )

    print(f"[INFO] User-Agent usado: {user_agent}")
    return driver

## 5. Simulación de interacción con la página

Se realizan pequeños movimientos del cursor y desplazamientos verticales para reproducir una interacción básica con la página antes de extraer la información.

In [ ]:
def simulate_human_behavior(driver):
    """Realiza movimientos del cursor y scroll aleatorios."""

    try:
        body = driver.find_element(By.TAG_NAME, "body")
        action = ActionChains(driver)

        # Mueve el cursor a diferentes posiciones de la página.
        for _ in range(random.randint(2, 4)):
            x_offset = random.randint(10, 300)
            y_offset = random.randint(10, 300)
            action.move_to_element_with_offset(
                body, x_offset, y_offset
            ).pause(random.uniform(0.5, 1.2))

        action.perform()

        # Realiza varios desplazamientos verticales.
        for _ in range(random.randint(2, 4)):
            scroll_amount = random.randint(200, 600)
            driver.execute_script(
                f"window.scrollBy(0, {scroll_amount});"
            )
            time.sleep(random.uniform(1.0, 2.5))

    except Exception as error:
        print(f"[WARN] Error al simular comportamiento: {error}")

## 6. Aceptación de cookies

El botón de cookies puede encontrarse directamente en la página o dentro de un `iframe`. Esta función intenta cubrir ambos casos.

In [ ]:
def accept_cookies(driver, timeout=3):
    """Intenta aceptar el aviso de cookies de la página."""

    try:
        wait = WebDriverWait(driver, timeout)
        iframes = driver.find_elements(By.TAG_NAME, "iframe")

        # Primero se busca el botón dentro de los iframes.
        for iframe in iframes:
            driver.switch_to.frame(iframe)
            try:
                button = wait.until(
                    EC.element_to_be_clickable(
                        (By.XPATH, "//button[@title='Accept Cookies']")
                    )
                )
                button.click()
                print("[INFO] Cookies aceptadas (iframe).")
                driver.switch_to.default_content()
                return True
            except Exception:
                driver.switch_to.default_content()

        # Si no apareció en un iframe, se busca en la página principal.
        button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(), 'Accept Cookies')]")
            )
        )
        button.click()
        print("[INFO] Cookies aceptadas (página principal).")
        return True

    except Exception:
        print("[INFO] No se encontró el botón de cookies.")
        return False

## 7. Búsqueda de artículos

Se recorren las páginas de resultados de búsqueda de Financial Times y se extraen el título y la URL de cada resultado.

In [ ]:
def scrape_ft(company, num_pages):
    """Busca artículos de una empresa y guarda sus URLs en un TXT."""

    data = []

    for page_num in range(1, num_pages + 1):
        driver = create_driver()

        try:
            search_url = (
                f"https://www.ft.com/search?q={company}"
                f"&page={page_num}&sort=relevance&isFirstView=true"
            )
            driver.get(search_url)
            driver.switch_to.window(driver.window_handles[0])

            # Espera inicial para que la página y la extensión terminen de cargar.
            time.sleep(random.uniform(3, 6))
            simulate_human_behavior(driver)
            accept_cookies(driver)

            wait = WebDriverWait(driver, 5)
            wait.until(
                EC.presence_of_element_located((By.ID, "site-content"))
            )
            print(f"[INFO] Página {page_num} cargada.")

            elements = driver.find_elements(
                By.CLASS_NAME, "search-results__list-item"
            )

            for index, element in enumerate(elements):
                try:
                    link = element.find_element(
                        By.XPATH, ".//a[@data-trackable='heading-link']"
                    )
                    href = link.get_attribute("href")
                    title = link.text.strip()

                    data.append({"title": title, "href": href})
                    print(f"[OK] {title} -> {href}")

                except Exception:
                    print(
                        f"[WARN] Elemento {index}: "
                        "no se encontró el enlace."
                    )

        except Exception as error:
            print(
                f"[ERROR] No se pudo procesar la página "
                f"{page_num}: {error}"
            )

        finally:
            # Asegura que el navegador se cierre incluso si ocurre un error.
            time.sleep(random.uniform(2, 4))
            driver.quit()

    # Guardamos todos los resultados en un único archivo de texto.
    output_file = f"{company}.txt"
    with open(output_file, "w", encoding="utf-8") as file:
        for item in data:
            file.write(f"Título: {item['title']}\n")
            file.write(f"URL: {item['href']}\n")
            file.write("\n" + "=" * 50 + "\n\n")

    print(f"\n[OK] Datos exportados a '{output_file}'")

## 8. Extracción de artículos

A partir del archivo TXT generado en la fase anterior, se visitan las URLs una a una. De cada artículo se intenta obtener título, resumen, imágenes, cuerpo y fecha de publicación.

In [ ]:
def extract_article_data(driver, url):
    """Extrae la información disponible de un artículo."""

    article_id = url.rstrip("/").split("/")[-1]
    author = random.choice(AUTHORS)

    # Estructura base que se convertirá posteriormente en JSON.
    content = {
        "id": article_id,
        "title": "",
        "slug": url,
        "author": {"id": author["id"], "name": author["name"]},
        "summary": "",
        "body": [],
        "multimedias": [],
        "publication": "",
        "status": "PUBLISHED",
        "creation": "",
        "views": random.randint(100, 5000),
        "version": random.randint(1, 10),
        "section": random.choice(SECTIONS),
        "categories": [
            {"name": random.choice(CATEGORIES)}
            for _ in range(random.randint(1, 3))
        ],
    }

    # ---------- Título y resumen ----------
    try:
        topper = driver.find_element(By.ID, "o-topper")
        paragraphs = topper.text.split("\n")

        if len(paragraphs) >= 2:
            content["title"] = paragraphs[-2]
            content["summary"] = paragraphs[-1]
    except Exception:
        print("[WARN] No se encontró el encabezado del artículo.")

    # ---------- Imágenes ----------
    try:
        images = driver.find_elements(By.TAG_NAME, "img")

        for image in images:
            source = image.get_attribute("src")

            if source:
                content["multimedias"].append({
                    "type": "IMAGE",
                    "format": "AVIF",
                    "url": source,
                    "description": random.choice(IMAGE_DESCRIPTIONS),
                })

    except Exception:
        print("[WARN] No se encontraron imágenes.")

    # ---------- Cuerpo del artículo ----------
    try:
        article_body = driver.find_element(By.ID, "article-body")
        full_text = article_body.text.strip()
        content["body"] = [full_text] if full_text else []
        print("[OK] Se encontró el body del artículo.")
    except Exception:
        print("[WARN] No se encontró el artículo principal.")

    # ---------- Fecha de publicación ----------
    try:
        time_element = driver.find_element(By.TAG_NAME, "time")
        publication = time_element.get_attribute("title")
        content["publication"] = publication

        # El código original utiliza una fecha de creación simulada,
        # entre 1 y 30 días anterior a la publicación.
        publication_date = datetime.datetime.strptime(
            publication,
            "%B %d %Y %I:%M %p",
        )
        creation_date = publication_date - datetime.timedelta(
            days=random.randint(1, 30)
        )
        content["creation"] = creation_date.strftime(
            "%B %d %Y %I:%M %p"
        )

    except Exception:
        print("[WARN] No se pudo obtener la fecha de publicación.")

    return article_id, content

## 9. Scraping y generación de JSON

Esta función lee el TXT de URLs, procesa cada artículo y guarda un JSON independiente en la carpeta correspondiente a la empresa.

In [ ]:
def scrape_articles(company):
    """Procesa las URLs de una empresa y genera un JSON por artículo."""

    input_file = f"{company}.txt"
    destination_folder = os.path.join(
        r"C:\repositorio\WebScrapping",
        company,
    )
    os.makedirs(destination_folder, exist_ok=True)

    # Leemos únicamente las líneas que contienen URLs.
    try:
        with open(input_file, "r", encoding="utf-8") as file:
            urls = [
                line.replace("URL: ", "").strip()
                for line in file
                if line.startswith("URL:")
            ]
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo {input_file}")
        return

    print(f"[INFO] Se encontraron {len(urls)} URLs.")

    for url in urls:
        print(f"\n=== PROCESANDO URL: {url} ===")
        driver = create_driver()

        try:
            driver.get(url)
            time.sleep(5)

            # Si la extensión abre más de una pestaña, se mantiene el foco
            # en la última pestaña disponible, como hacía el código original.
            if len(driver.window_handles) > 1:
                driver.switch_to.window(driver.window_handles[-1])

            simulate_human_behavior(driver)
            accept_cookies(driver)

            try:
                driver.maximize_window()
            except Exception:
                pass

            simulate_human_behavior(driver)

            # Extraemos los datos y los guardamos como JSON.
            article_id, content = extract_article_data(driver, url)
            json_path = os.path.join(
                destination_folder,
                f"{article_id}.json",
            )

            with open(json_path, "w", encoding="utf-8") as file:
                json.dump(
                    content,
                    file,
                    indent=4,
                    ensure_ascii=False,
                )

            print(f"[OK] Guardado archivo: {json_path}")

        except Exception as error:
            print(f"[ERROR] No se pudo procesar el artículo: {error}")

        finally:
            # Cerramos siempre el navegador para evitar procesos de Chrome
            # abiertos si alguna extracción falla.
            time.sleep(random.uniform(2, 4))
            driver.quit()

## 10. Ejecución

Ejemplo para procesar artículos relacionados con Tesla.

Primero se puede ejecutar la búsqueda para generar el archivo `Tesla.txt`. Después se ejecuta el scraper de artículos para crear los JSON.

In [ ]:
# Ejemplo:
# scrape_ft("Tesla", num_pages=3)
# scrape_articles("Tesla")

### Cambios realizados respecto al notebook original

- Se eliminaron imports repetidos.
- Se eliminaron las tres versiones duplicadas de `generate_user_agents()`.
- Se consolidaron las distintas versiones de `create_driver()`.
- Se consolidó `simulate_human_behavior()` en una única función.
- Se separó la aceptación de cookies en una función reutilizable.
- Se separó la extracción de datos de un artículo de la gestión del scraping.
- Se añadieron comentarios y docstrings en las funciones principales.
- Se agruparon las constantes y datos simulados al principio del notebook.
- Se utilizó `try/finally` para garantizar el cierre de Chrome.
- Se mantuvo la lógica general y la estructura de datos del código original.